## Usage notes

- You can load data directly from W&B or from local CSVs produced by `scripts/hyperparam_search.py`.
- For W&B access, set `$WANDB_API_KEY` or log in once via `wandb.login()`.
- The primary selection metric is `summary.best_val/f1_macro` (mean across seeds).

If you only have partial runs, the notebook will still compute partial summaries.

In [ ]:
import os
import json
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml

try:
    import wandb
except Exception:
    wandb = None

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

In [ ]:
# ===== User parameters =====
ENTITY = "bioshape-lab"
PROJECTS = [
    "bgbench_multi_dataset_grid_search",
    "bgbench_multi_dataset_grid_search_readout",
]

USE_WANDB = True  # set False to load from local CSVs only

ROOT = Path(".").resolve()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent

LOCAL_RESULT_GLOBS = [
    str(ROOT / "search_results/**/results_train_*.csv"),
    str(ROOT / "search_results/**/results_*.csv"),
]

METRIC_KEYS = [
    "summary.best_val/f1_macro",
    "summary.best_val_f1_macro",
    "best_val/f1_macro",
    "best_val_f1_macro",
]
USE_HISTORY_IF_MISSING = False
HISTORY_KEYS = [
    "summary.val/f1_macro",
    "summary.best_val/f1_macro",
    "val/f1_macro",
    "best_val/f1_macro",
]

MIN_SEEDS_REQUIRED = 3  # used when ranking configs; set 3 for strict

CONFIG_YAMLS = [
    str(ROOT / "configs/hparams_search/multi_dataset_grid_search.yaml"),
    str(ROOT / "configs/hparams_search/multi_dataset_grid_search_readout.yaml"),
]

OUTPUT_DIR = ROOT / "notebooks/analysis_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def flatten_dict(d: dict[str, Any], parent_key: str = "", sep: str = ".") -> dict[str, Any]:
    items: dict[str, Any] = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items


def normalize_config(config: dict[str, Any]) -> dict[str, Any]:
    flat = flatten_dict(config)
    return {k: v for k, v in flat.items() if not k.startswith("_")}


def first_present(d: dict[str, Any], keys: Iterable[str]) -> Any:
    for key in keys:
        if key in d and d[key] is not None:
            return d[key]
    return None


def coerce_list(value: Any) -> Any:
    if isinstance(value, (list, tuple)):
        return tuple(value)
    return value


def safe_to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="ignore")

In [ ]:
def load_grid_keys(config_paths: list[str]) -> list[str]:
    keys: set[str] = set()
    for path in config_paths:
        with open(path, "r") as f:
            cfg = yaml.safe_load(f)

        for section in ["shared_grid", "per_model_grid", "per_model_dataset_grid", "per_dataset_ratio_method_grid"]:
            if section not in cfg:
                continue
            if section == "shared_grid":
                keys.update(cfg[section].keys())
            elif section == "per_model_grid":
                for model_grid in cfg[section].values():
                    keys.update(model_grid.keys())
            elif section == "per_model_dataset_grid":
                for md_grid in cfg[section].values():
                    keys.update(md_grid.keys())
            elif section == "per_dataset_ratio_method_grid":
                for drm_grid in cfg[section].values():
                    keys.update(drm_grid.keys())

    keys.update(["model", "dataset", "seed"])
    return sorted(keys)


GRID_KEYS = load_grid_keys(CONFIG_YAMLS)
GRID_KEYS[:10], len(GRID_KEYS)

In [ ]:
def load_wandb_runs(
    entity: str,
    projects: list[str],
    metric_keys: list[str],
    use_history_if_missing: bool = False,
    history_keys: list[str] | None = None,
    max_runs: int | None = None,
) -> pd.DataFrame:
    if wandb is None:
        raise RuntimeError("wandb is not available in this environment.")

    api = wandb.Api()
    all_rows = []

    for project in projects:
        runs = api.runs(f"{entity}/{project}")
        if max_runs:
            runs = runs[:max_runs]

        for run in runs:
            config = normalize_config(run.config or {})
            summary = normalize_config(run.summary or {})

            metric_value = first_present(summary, metric_keys)

            if metric_value is None and use_history_if_missing and history_keys:
                try:
                    history = run.history(keys=history_keys, pandas=True)
                    if not history.empty:
                        metric_value = history[history_keys].max(numeric_only=True).max()
                except Exception:
                    metric_value = None

            row = {
                "run_id": run.id,
                "project": project,
                "name": run.name,
                "state": run.state,
                "tags": ";".join(run.tags or []),
                "metric_value": metric_value,
                **config,
                **{f"summary.{k}": v for k, v in summary.items()},
            }
            all_rows.append(row)

    df = pd.DataFrame(all_rows)
    return df


def load_local_results(globs: list[str]) -> pd.DataFrame:
    paths = []
    for pattern in globs:
        paths.extend(Path(".").glob(pattern))

    frames = []
    for path in sorted(set(paths)):
        df = pd.read_csv(path)
        df["source"] = str(path)
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

In [ ]:
if USE_WANDB:
    if wandb is None:
        raise RuntimeError("wandb is not installed. Please install it or set USE_WANDB=False.")
    if not os.getenv("WANDB_API_KEY"):
        print("WANDB_API_KEY not set. If needed, run wandb.login() once.")
    wandb_df = load_wandb_runs(
        ENTITY,
        PROJECTS,
        METRIC_KEYS,
        use_history_if_missing=USE_HISTORY_IF_MISSING,
        history_keys=HISTORY_KEYS,
    )
else:
    wandb_df = pd.DataFrame()

local_df = load_local_results(LOCAL_RESULT_GLOBS)

wandb_df.shape, local_df.shape

In [ ]:
def normalize_metric_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "metric_value" in df.columns and df["metric_value"].notna().any():
        return df
    for key in METRIC_KEYS:
        if key in df.columns:
            df["metric_value"] = df[key]
            return df
        summary_key = f"summary.{key}"
        if summary_key in df.columns:
            df["metric_value"] = df[summary_key]
            return df
    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "model" not in df.columns:
        if "model.name" in df.columns:
            df["model"] = df["model.name"]
        elif "model.model_name" in df.columns:
            df["model"] = df["model.model_name"]
    if "dataset" not in df.columns:
        if "data.name" in df.columns:
            df["dataset"] = df["data.name"]
        elif "dataset.loader.parameters.data_name" in df.columns:
            df["dataset"] = df["dataset.loader.parameters.data_name"]
    return df


def unify_sources(wandb_df: pd.DataFrame, local_df: pd.DataFrame) -> pd.DataFrame:
    frames = []
    if not wandb_df.empty:
        frames.append(wandb_df)
    if not local_df.empty:
        frames.append(local_df)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df = normalize_metric_column(df)
    df = standardize_columns(df)
    if "seed" in df.columns:
        df["seed"] = safe_to_numeric(df["seed"])
    df["metric_value"] = pd.to_numeric(df["metric_value"], errors="coerce")
    return df


raw_df = unify_sources(wandb_df, local_df)
raw_df.shape

## Clean and filter runs
We keep only successful runs with a valid `metric_value` and the required columns.

In [ ]:
df = raw_df.copy()

required_cols = ["model", "dataset", "seed"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[df["metric_value"].notna()].copy()
df = df[df["metric_value"] > -np.inf]

df["seed"] = safe_to_numeric(df["seed"])
df = df.drop_duplicates(subset=["run_id"] if "run_id" in df.columns else None)

df.shape

In [ ]:
df["seed"]

## Build configuration IDs (hyperparameter signature)
We use the hyperparameters defined in the grid YAMLs to build a stable configuration signature.

In [ ]:
def build_config_id(row: pd.Series, keys: list[str]) -> str:
    parts = []
    for key in keys:
        if key in row.index:
            value = coerce_list(row[key])
            parts.append(f"{key}={value}")
    return "|".join(parts)

excluded_keys = {"seed", "model", "dataset"}
available_keys = [k for k in GRID_KEYS if k in df.columns and k not in excluded_keys]
df["config_id"] = df.apply(build_config_id, axis=1, keys=available_keys)

len(available_keys), available_keys[:12]

In [ ]:
# Diagnostics: how many seeds per model/dataset (before config grouping)?
(df.groupby(["model", "dataset"])
   .agg(n_runs=("metric_value", "size"), n_seeds=("seed", "nunique"))
   .reset_index()
   .sort_values(["model", "dataset"])
   .head(20))

## Aggregate over seeds
Compute mean/std of `summary.best_val/f1_macro` across seeds for each configuration.

In [ ]:
group_keys = ["model", "dataset", "config_id"]
agg = (
    df.groupby(group_keys)
    .agg(
        metric_mean=("metric_value", "mean"),
        metric_std=("metric_value", "std"),
        n_seeds=("seed", "nunique"),
    )
    .reset_index()
)

# attach hyperparameter columns (first value per config)
hp_cols = [k for k in available_keys if k not in group_keys]
hp_first = df.groupby(group_keys)[hp_cols].first().reset_index()
agg = agg.merge(hp_first, on=group_keys, how="left")

agg = agg[agg["n_seeds"] >= MIN_SEEDS_REQUIRED]
agg.sort_values(["model", "dataset", "metric_mean"], ascending=[True, True, False]).head()

In [ ]:
for elem in df.columns:
    print(elem)

## Best configuration per model and dataset

In [ ]:
best_by_model_dataset = (
    agg.sort_values(["model", "dataset", "metric_mean"], ascending=[True, True, False])
    .groupby(["model", "dataset"])
    .head(1)
    .reset_index(drop=True)
)

best_by_model_dataset[["model", "dataset", "metric_mean", "metric_std", "n_seeds"] + hp_cols].head(10)

## Best configuration per dataset (any model)

In [ ]:
best_by_dataset = (
    agg.sort_values(["dataset", "metric_mean"], ascending=[True, False])
    .groupby(["dataset"])
    .head(1)
    .reset_index(drop=True)
)

best_by_dataset[["model", "dataset", "metric_mean", "metric_std", "n_seeds"] + hp_cols]

## Seed stability

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=agg, x="metric_mean", y="metric_std", hue="model", style="dataset", alpha=0.8)
plt.title("Seed stability: mean vs std")
plt.xlabel("Mean summary.best_val/f1_macro")
plt.ylabel("Std across seeds")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

## Marginal hyperparameter effects
We compute average performance per hyperparameter value, optionally split by model and dataset.

In [ ]:
def plot_marginal_effect(df: pd.DataFrame, key: str, by: list[str] | None = None):
    if key not in df.columns:
        print(f"Skipping {key} (not present).")
        return
    group_cols = [key] + (by or [])
    summary = (
        df.groupby(group_cols)["metric_value"].mean().reset_index()
    )
    plt.figure(figsize=(7, 4))
    if by:
        sns.lineplot(data=summary, x=key, y="metric_value", hue=by[0], style=by[1] if len(by) > 1 else None)
    else:
        sns.barplot(data=summary, x=key, y="metric_value")
    plt.title(f"Marginal effect of {key}")
    plt.ylabel("Mean summary.best_val/f1_macro")
    plt.tight_layout()


candidate_keys = [
    "dataset.loader.parameters.node_sample_ratio",
    "dataset.loader.parameters.method",
    "optimizer.parameters.lr",
    "optimizer.parameters.weight_decay",
    "model.backbone.dropout",
    "model.readout.readout_name",
    "model.backbone.num_layers",
    "model.backbone.heads",
    "model.backbone.num_heads",
    "model.feature_encoder.out_channels",
]

for k in candidate_keys:
    plot_marginal_effect(df, k, by=["model", "dataset"])

## Ratio × method heatmaps
We take the best mean score per ratio/method for each model/dataset by maximizing over other hyperparameters.

In [ ]:
ratio_key = "dataset.loader.parameters.node_sample_ratio"
method_key = "dataset.loader.parameters.method"

if ratio_key in agg.columns and method_key in agg.columns:
    ratio_method = (
        agg.groupby(["model", "dataset", ratio_key, method_key])["metric_mean"]
        .max()
        .reset_index()
    )

    for (model, dataset), subset in ratio_method.groupby(["model", "dataset"]):
        pivot = subset.pivot(index=ratio_key, columns=method_key, values="metric_mean")
        plt.figure(figsize=(7, 4))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
        plt.title(f"{model} | {dataset}: best mean by ratio/method")
        plt.tight_layout()
else:
    print("Ratio/method keys not found in data.")

## Model selection across datasets
Aggregate over datasets to identify consistently strong models.

In [ ]:
model_summary = (
    agg.groupby(["model"])
    .agg(mean_score=("metric_mean", "mean"), std_score=("metric_mean", "std"), n_configs=("config_id", "nunique"))
    .reset_index()
)
model_summary.sort_values("mean_score", ascending=False)

## Feature importance (optional)
Train a quick model to estimate which hyperparameters are most predictive of performance.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

feature_cols = [k for k in available_keys if k in df.columns]
model_df = df[feature_cols + ["metric_value"]].dropna()

categorical_cols = [c for c in feature_cols if model_df[c].dtype == object]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf = RandomForestRegressor(n_estimators=200, random_state=42)
pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", rf)])

if not model_df.empty:
    pipe.fit(model_df[feature_cols], model_df["metric_value"])
    # Extract feature names after one-hot encoding
    feature_names = list(pipe.named_steps["preprocess"].get_feature_names_out())
    importances = pipe.named_steps["model"].feature_importances_

    imp_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(30)
    )

    plt.figure(figsize=(8, 6))
    sns.barplot(data=imp_df, y="feature", x="importance")
    plt.title("Top feature importances")
    plt.tight_layout()
else:
    print("No data available for feature importance.")

## Export summary tables

In [ ]:
best_by_model_dataset.to_csv(OUTPUT_DIR / "best_by_model_dataset.csv", index=False)
best_by_dataset.to_csv(OUTPUT_DIR / "best_by_dataset.csv", index=False)
model_summary.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)

print("Saved outputs to", OUTPUT_DIR)